In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/ad9190/datasets-lsb/features_train_gray_lsb_payloads.csv
/kaggle/input/datasets/ad9190/datasets-lsb/features_train_gray.csv


In [2]:
import os
import warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils import resample
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    precision_score, recall_score, matthews_corrcoef, confusion_matrix,
)

warnings.filterwarnings("ignore", message="X does not have valid feature names")

# ── Configuration ─────────────────────────────────────────────────────────────
COVER_CSV    = "/kaggle/input/datasets/ad9190/datasets-lsb/features_train_gray.csv"
STEGO_CSV    = "/kaggle/input/datasets/ad9190/datasets-lsb/features_train_gray_lsb_payloads.csv"
DATA_DIR     = "/kaggle/working/data"
MODELS_DIR   = "/kaggle/working/models"
RESULTS_DIR  = "/kaggle/working/results"
RANDOM_STATE = 42
MODEL_ORDER = ["LR", "RF", "XGB", "LGBM", "MLP"]
REDUNDANT_FEATURES = {"Adjacent Bin Difference", "Pairwise Hist Diff Sum"}

os.makedirs(DATA_DIR,    exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)



In [3]:
# ── Step 1 — Load & Build Dataset ─────────────────────────────────────────────
cover_cols = pd.read_csv(COVER_CSV, nrows=0).columns.tolist()
stego_cols = pd.read_csv(STEGO_CSV, nrows=0).columns.tolist()
feature_cols = [c for c in cover_cols if c != "ImageFile"]

if len(feature_cols) != 41:
    warnings.warn(f"Expected 41 feature columns, found {len(feature_cols)}.", RuntimeWarning)
missing = [c for c in feature_cols if c not in stego_cols]
if missing:
    raise ValueError(f"Stego CSV missing columns: {missing}")

dtype_map       = {c: np.float32 for c in feature_cols}
dtype_map_stego = {**dtype_map, "Payload": np.float32}

cover_df = pd.read_csv(COVER_CSV, usecols=["ImageFile", *feature_cols], dtype=dtype_map)
cover_df.insert(1, "Payload", np.float32(0.0))
cover_df.insert(2, "Label",   np.uint8(0))

stego_df = pd.read_csv(STEGO_CSV, usecols=["ImageFile", "Payload", *feature_cols], dtype=dtype_map_stego)
stego_df.insert(2, "Label", np.uint8(1))

full_df = pd.concat([cover_df, stego_df], axis=0, ignore_index=True)
full_df["GroupKey"] = full_df["ImageFile"].apply(
    lambda p: os.path.basename(str(p).strip().replace("\\", "/")).lower()
)


In [4]:

# ── Step 2 — Group Split 70 / 15 / 15 ─────────────────────────────────────────
sp1 = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=RANDOM_STATE)
train_idx, temp_idx = next(sp1.split(full_df, groups=full_df["GroupKey"]))
train_df = full_df.iloc[train_idx].copy()
temp_df  = full_df.iloc[temp_idx].copy()

sp2 = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=RANDOM_STATE)
val_idx, test_idx = next(sp2.split(temp_df, groups=temp_df["GroupKey"]))
val_df  = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

assert not (set(train_df["GroupKey"]) & set(val_df["GroupKey"])),  "Train/val leakage!"
assert not (set(train_df["GroupKey"]) & set(test_df["GroupKey"])), "Train/test leakage!"
assert not (set(val_df["GroupKey"])   & set(test_df["GroupKey"])), "Val/test leakage!"

for df, fname in [(train_df, "train.csv"), (val_df, "validate.csv"), (test_df, "test.csv")]:
    df[["ImageFile", "Payload", "Label", *feature_cols]].to_csv(
        os.path.join(DATA_DIR, fname), index=False
    )

print(f"Step 1-2 done | train {len(train_df):,} / val {len(val_df):,} / test {len(test_df):,} rows")



Step 1-2 done | train 812,834 / val 173,582 / test 172,822 rows


In [5]:
# ── Step 3 — Preprocessing + Balanced Training Set ────────────────────────────
selected_cols = [c for c in feature_cols if c not in REDUNDANT_FEATURES]
print(f"Training features: {len(selected_cols)} (dropped {len(REDUNDANT_FEATURES)} redundant)")

def to_xy(df):
    return (
        df[selected_cols].to_numpy(dtype=np.float32),
        df["Label"].to_numpy(dtype=np.uint8),
    )

# Undersample stego to match clean count — fixes XGB/LGBM class collapse
train_clean      = train_df[train_df["Label"] == 0]
train_stego      = train_df[train_df["Label"] == 1]
train_stego_down = resample(
    train_stego,
    n_samples=len(train_clean),
    random_state=RANDOM_STATE,
    replace=False,
)
train_balanced = pd.concat([train_clean, train_stego_down]).sample(
    frac=1, random_state=RANDOM_STATE
)
print(f"Balanced training set: {len(train_balanced):,} rows "
      f"({len(train_clean):,} clean / {len(train_clean):,} stego)")

x_train_raw, y_train = to_xy(train_balanced)
x_val_raw,   y_val   = to_xy(val_df)
x_test_raw,  y_test  = to_xy(test_df)

scaler         = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train_raw)
x_val_scaled   = scaler.transform(x_val_raw)
x_test_scaled  = scaler.transform(x_test_raw)

print("Step 3 done | preprocessing complete")



Training features: 39 (dropped 2 redundant)
Balanced training set: 165,588 rows (82,794 clean / 82,794 stego)
Step 3 done | preprocessing complete


In [6]:
dropped_cols = [c for c in feature_cols if c in REDUNDANT_FEATURES]
print("Dropped columns:")
for col in dropped_cols:
    print(col)

Dropped columns:
Adjacent Bin Difference
Pairwise Hist Diff Sum


In [ ]:
# ── Step 4 — Train Models ──────────────────────────────────────────────────────
models = {}

# Logistic Regression
lr = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE
    )),
])
lr.fit(x_train_raw, y_train)
joblib.dump(lr, os.path.join(MODELS_DIR, "lr_pipeline.joblib"))
models["LR"] = lr
print("Trained: Logistic Regression")

# Random Forest
rf = RandomForestClassifier(
    n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE
)
rf.fit(x_train_scaled, y_train)
joblib.dump(rf, os.path.join(MODELS_DIR, "rf_model.joblib"))
models["RF"] = rf
print("Trained: Random Forest")

# XGBoost — no scale_pos_weight, training data is already balanced
try:
    from xgboost import XGBClassifier
    xgb = XGBClassifier(
        n_estimators=500,
        eval_metric="logloss",
        early_stopping_rounds=20,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    xgb.fit(x_train_scaled, y_train, eval_set=[(x_val_scaled, y_val)], verbose=False)
    joblib.dump(xgb, os.path.join(MODELS_DIR, "xgb_model.joblib"))
    models["XGB"] = xgb
    print("Trained: XGBoost")
except ImportError:
    warnings.warn("xgboost not installed — skipping.", RuntimeWarning)
except Exception as e:
    warnings.warn(f"XGBoost failed: {e} — skipping.", RuntimeWarning)

# LightGBM — no is_unbalance, training data is already balanced
try:
    import lightgbm as lgb
    lgbm = lgb.LGBMClassifier(
        n_estimators=500,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    try:
        lgbm.fit(
            x_train_scaled, y_train,
            eval_set=[(x_val_scaled, y_val)],
            eval_metric="binary_logloss",
            callbacks=[lgb.early_stopping(20, verbose=False)],
        )
    except TypeError:
        lgbm.fit(
            x_train_scaled, y_train,
            eval_set=[(x_val_scaled, y_val)],
            eval_metric="binary_logloss",
        )
    joblib.dump(lgbm, os.path.join(MODELS_DIR, "lgbm_model.joblib"))
    models["LGBM"] = lgbm
    print("Trained: LightGBM")
except ImportError:
    warnings.warn("lightgbm not installed — skipping.", RuntimeWarning)
except Exception as e:
    warnings.warn(f"LightGBM failed: {e} — skipping.", RuntimeWarning)



In [ ]:
# MLP Neural Network
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu",
    solver="adam",
    batch_size=512,
    learning_rate="adaptive",
    max_iter=200,          # increased from 100
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,   # increased from 10 — more patient
    tol=1e-5,              # tighter than default 1e-4
    random_state=RANDOM_STATE,
    verbose=True,
)
mlp.fit(x_train_scaled, y_train)
joblib.dump(mlp, os.path.join(MODELS_DIR, "mlp_model.joblib"))
models["MLP"] = mlp
print("Trained: MLP Neural Network")

In [ ]:
# ── Step 5 — Evaluate ─────────────────────────────────────────────────────────
def predict(key, model, x_raw, x_scaled):
    x = x_raw if key == "LR" else x_scaled
    y_pred = model.predict(x)
    y_prob = (
        model.predict_proba(x)[:, 1]
        if hasattr(model, "predict_proba")
        else model.decision_function(x)
    )
    return y_pred, y_prob

def overall_metrics(y_true, y_pred, y_prob):
    try:    auc = float(roc_auc_score(y_true, y_prob))
    except: auc = np.nan
    return {
        "Accuracy":        float(accuracy_score(y_true, y_pred)),
        "ROC_AUC":         auc,
        "F1_Macro":        float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "Precision_Macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "Recall_Macro":    float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "MCC":             float(matthews_corrcoef(y_true, y_pred)),
    }

def per_payload_acc(key, model):
    payloads   = sorted([float(p) for p in test_df.loc[test_df["Label"] == 1, "Payload"].unique() if p > 0])
    clean_pool = test_df[test_df["Payload"] == 0.0]
    acc = {}
    for p in payloads:
        stego_sub = test_df[test_df["Payload"] == p]
        clean_sub = clean_pool.sample(n=len(stego_sub), random_state=RANDOM_STATE)
        sub       = pd.concat([clean_sub, stego_sub])
        xr, yt    = to_xy(sub)
        yp, _     = predict(key, model, xr, scaler.transform(xr))
        acc[p]    = float(accuracy_score(yt, yp))
    return acc

def band_mean(acc, lo, hi):
    v = [a for p, a in acc.items() if lo <= p <= hi and not np.isnan(a)]
    return float(np.mean(v)) if v else np.nan

def detect_threshold(acc, t=0.70):
    for i, p in enumerate(sorted(acc)):
        if not np.isnan(acc[p]) and acc[p] > t:
            tail = [acc[q] for q in sorted(acc)[i:] if not np.isnan(acc[q])]
            if tail and all(v >= t for v in tail):
                return float(p)
    return np.nan

payload_values = sorted([
    float(p) for p in test_df.loc[test_df["Label"] == 1, "Payload"].unique() if p > 0
])
per_payload_df = pd.DataFrame({"Payload": payload_values})
metrics_rows, threshold_rows, conf_mats = [], [], {}

for key in MODEL_ORDER:
    model = models.get(key)
    if model is None:
        metrics_rows.append({"Model": key, "Status": "skipped"})
        threshold_rows.append({"Model": key, "DetectionThresholdPayload": np.nan})
        per_payload_df[key] = np.nan
        continue

    y_pred, y_prob = predict(key, model, x_test_raw, x_test_scaled)
    om  = overall_metrics(y_test, y_pred, y_prob)
    ppa = per_payload_acc(key, model)

    metrics_rows.append({
        "Model": key, **om,
        "LowBandMeanAcc":  band_mean(ppa, 0.05, 0.50),
        "MidBandMeanAcc":  band_mean(ppa, 0.55, 1.00),
        "HighBandMeanAcc": band_mean(ppa, 1.05, 2.00),
        "Status": "ok",
    })
    threshold_rows.append({"Model": key, "DetectionThresholdPayload": detect_threshold(ppa)})
    per_payload_df[key] = per_payload_df["Payload"].map(lambda p: ppa.get(float(p), np.nan))
    conf_mats[key] = confusion_matrix(y_test, y_pred, labels=[0, 1])
    print(f"Evaluated: {key} | Accuracy {om['Accuracy']:.4f} | AUC {om['ROC_AUC']:.4f} | "
          f"F1 {om['F1_Macro']:.4f} | MCC {om['MCC']:.4f}")



In [ ]:
# ── Step 6 — Save Results ──────────────────────────────────────────────────────
pd.DataFrame(metrics_rows).to_csv(
    os.path.join(RESULTS_DIR, "metrics_summary.csv"), index=False)
per_payload_df.to_csv(
    os.path.join(RESULTS_DIR, "per_payload_accuracy.csv"), index=False)
pd.DataFrame(threshold_rows).to_csv(
    os.path.join(RESULTS_DIR, "detection_thresholds.csv"), index=False)

# Confusion matrices
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for ax, key in zip(axes.ravel(), MODEL_ORDER):
    if key not in conf_mats:
        ax.axis("off")
        ax.set_title(f"{key} (skipped)")
        continue
    cm = conf_mats[key]
    ax.imshow(cm, cmap="Blues")
    ax.set_title(key)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    for r in range(2):
        for c in range(2):
            ax.text(c, r, str(cm[r, c]), ha="center", va="center", color="black")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "confusion_matrices.png"), dpi=220)
plt.show()

# Per-payload accuracy plot
fig, ax = plt.subplots(figsize=(11, 6))
for key in MODEL_ORDER:
    if key in per_payload_df.columns:
        ax.plot(per_payload_df["Payload"], per_payload_df[key], marker="o", label=key)
ax.axhline(0.70, linestyle="--", color="red", label="70% threshold")
ax.set_xlabel("Payload (bpp)")
ax.set_ylabel("Accuracy")
ax.set_title("Per-Payload Detection Accuracy")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "per_payload_accuracy.png"), dpi=220)
plt.show()

print("\nAll done. Outputs saved to /kaggle/working/")

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression as MetaLR
from sklearn.ensemble import StackingClassifier
from xgboost import XGBClassifier as XGBStack
import lightgbm as lgb

# Fresh base learners WITHOUT early stopping — for stacking only
stack_rf = RandomForestClassifier(
    n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE
)
stack_xgb = XGBStack(
    n_estimators=300,        # fixed trees, no early stopping
    eval_metric="logloss",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
stack_lgbm = lgb.LGBMClassifier(
    n_estimators=300,        # fixed trees, no early stopping
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

stack_name = "Stack_RF_XGB_LGBM__LR"

stack = StackingClassifier(
    estimators=[
        ("rf",   stack_rf),
        ("xgb",  stack_xgb),
        ("lgbm", stack_lgbm),
    ],
    final_estimator=MetaLR(max_iter=1000, random_state=RANDOM_STATE),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    stack_method="predict_proba",
    passthrough=False,
    n_jobs=-1,
    verbose=1,
)

stack.fit(x_train_scaled, y_train)
joblib.dump(stack, os.path.join(MODELS_DIR, f"{stack_name}.joblib"))
models[stack_name] = stack
print(f"Trained: {stack_name}")

# Evaluate
y_pred_s, y_prob_s = predict(stack_name, stack, x_test_raw, x_test_scaled)
om_s = overall_metrics(y_test, y_pred_s, y_prob_s)
ppa_s = per_payload_acc(stack_name, stack)

print(f"\n{stack_name}")
print(f"  Accuracy {om_s['Accuracy']:.4f} | AUC {om_s['ROC_AUC']:.4f} | "
      f"F1 {om_s['F1_Macro']:.4f} | MCC {om_s['MCC']:.4f}")
print(f"  Low band  (0.05-0.50): {band_mean(ppa_s, 0.05, 0.50):.4f}")
print(f"  Mid band  (0.55-1.00): {band_mean(ppa_s, 0.55, 1.00):.4f}")
print(f"  High band (1.05-2.00): {band_mean(ppa_s, 1.05, 2.00):.4f}")
print(f"  Detection threshold:   {detect_threshold(ppa_s):.2f} bpp")

# Save per-payload CSV
trial1_df = pd.DataFrame({
    "Payload":  sorted(ppa_s.keys()),
    "Accuracy": [ppa_s[p] for p in sorted(ppa_s.keys())]
})
trial1_df.to_csv(os.path.join(RESULTS_DIR, f"{stack_name}_per_payload.csv"), index=False)

# Append to master CSV
per_payload_df[stack_name] = per_payload_df["Payload"].map(
    lambda p: ppa_s.get(float(p), np.nan)
)
per_payload_df.to_csv(os.path.join(RESULTS_DIR, "per_payload_accuracy.csv"), index=False)
print("CSVs saved.")

In [ ]:
# ── Stacking Trials 2, 3, 4 ───────────────────────────────────────────────────
# Run each trial in a separate cell, one at a time.
# Requires: models, x_train_scaled, y_train, x_test_raw, x_test_scaled,
#           y_test, test_df, scaler, selected_cols, per_payload_df
#           to already be defined from the main pipeline cell.

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression as MetaLR
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.exceptions import DataConversionWarning
from xgboost import XGBClassifier as XGBStack
import lightgbm as lgb
import warnings

warnings.filterwarnings("ignore", message="X does not have valid feature names")

# Shared fresh base learners (no early stopping) — reused across all trials
stack_rf = RandomForestClassifier(
    n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE
)
stack_xgb = XGBStack(
    n_estimators=300,
    eval_metric="logloss",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
stack_lgbm = lgb.LGBMClassifier(
    n_estimators=300,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

def run_trial(stack_name, stack, x_train, y_train, x_test_raw, x_test_scaled, y_test):
    """Train, evaluate, save model and CSVs for a single stacking trial."""

    print(f"\n{'='*60}")
    print(f"Running: {stack_name}")
    print(f"{'='*60}")

    stack.fit(x_train, y_train)
    joblib.dump(stack, os.path.join(MODELS_DIR, f"{stack_name}.joblib"))
    models[stack_name] = stack
    print(f"Trained: {stack_name}")

    # Overall metrics
    y_pred_s, y_prob_s = predict(stack_name, stack, x_test_raw, x_test_scaled)
    om_s  = overall_metrics(y_test, y_pred_s, y_prob_s)
    ppa_s = per_payload_acc(stack_name, stack)

    print(f"\nResults — {stack_name}")
    print(f"  Accuracy  : {om_s['Accuracy']:.4f}")
    print(f"  AUC       : {om_s['ROC_AUC']:.4f}")
    print(f"  F1 Macro  : {om_s['F1_Macro']:.4f}")
    print(f"  MCC       : {om_s['MCC']:.4f}")
    print(f"  Low  band (0.05-0.50) : {band_mean(ppa_s, 0.05, 0.50):.4f}")
    print(f"  Mid  band (0.55-1.00) : {band_mean(ppa_s, 0.55, 1.00):.4f}")
    print(f"  High band (1.05-2.00) : {band_mean(ppa_s, 1.05, 2.00):.4f}")
    print(f"  Detection threshold   : {detect_threshold(ppa_s):.2f} bpp")

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred_s, labels=[0, 1])
    print(f"\n  Confusion Matrix:")
    print(f"    TN={cm[0,0]:>6}  FP={cm[0,1]:>6}")
    print(f"    FN={cm[1,0]:>6}  TP={cm[1,1]:>6}")

    # Per-payload CSV (standalone)
    trial_df = pd.DataFrame({
        "Payload":  sorted(ppa_s.keys()),
        "Accuracy": [ppa_s[p] for p in sorted(ppa_s.keys())]
    })
    standalone_path = os.path.join(RESULTS_DIR, f"{stack_name}_per_payload.csv")
    trial_df.to_csv(standalone_path, index=False)
    print(f"\n  Standalone CSV saved: {standalone_path}")

    # Append column to master per_payload_df and overwrite
    per_payload_df[stack_name] = per_payload_df["Payload"].map(
        lambda p: ppa_s.get(float(p), np.nan)
    )
    master_path = os.path.join(RESULTS_DIR, "per_payload_accuracy.csv")
    per_payload_df.to_csv(master_path, index=False)
    print(f"  Master CSV updated   : {master_path}")

    return om_s, ppa_s




In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TRIAL 2 — RF + XGB + LGBM + MLP → LR
# Run this cell on its own after Trial 1 completes
# ══════════════════════════════════════════════════════════════════════════════

stack_name_2 = "Stack_RF_XGB_LGBM_MLP__LR"

stack2 = StackingClassifier(
    estimators=[
        ("rf",   stack_rf),
        ("xgb",  stack_xgb),
        ("lgbm", stack_lgbm),
        ("mlp",  models["MLP"]),      # MLP already trained, used as-is
    ],
    final_estimator=MetaLR(max_iter=1000, random_state=RANDOM_STATE),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    stack_method="predict_proba",
    passthrough=False,
    n_jobs=-1,
    verbose=1,
)

om2, ppa2 = run_trial(
    stack_name_2, stack2,
    x_train_scaled, y_train,
    x_test_raw, x_test_scaled, y_test
)




In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TRIAL 3 — RF + XGB + LGBM → XGB meta-learner
# Run this cell on its own after Trial 2 completes
# ══════════════════════════════════════════════════════════════════════════════

stack_name_3 = "Stack_RF_XGB_LGBM__XGB"

stack3 = StackingClassifier(
    estimators=[
        ("rf",   stack_rf),
        ("xgb",  stack_xgb),
        ("lgbm", stack_lgbm),
    ],
    final_estimator=XGBStack(
        n_estimators=100,
        eval_metric="logloss",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    stack_method="predict_proba",
    passthrough=False,
    n_jobs=-1,
    verbose=1,
)

om3, ppa3 = run_trial(
    stack_name_3, stack3,
    x_train_scaled, y_train,
    x_test_raw, x_test_scaled, y_test
)



In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# TRIAL 4 — RF + XGB + LGBM → LR with passthrough
# (raw features also passed to meta-learner alongside base probabilities)
# Run this cell on its own after Trial 3 completes
# ══════════════════════════════════════════════════════════════════════════════

stack_name_4 = "Stack_RF_XGB_LGBM__LR_passthrough"

stack4 = StackingClassifier(
    estimators=[
        ("rf",   stack_rf),
        ("xgb",  stack_xgb),
        ("lgbm", stack_lgbm),
    ],
    final_estimator=MetaLR(max_iter=1000, random_state=RANDOM_STATE),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    stack_method="predict_proba",
    passthrough=True,         # key difference vs Trial 1
    n_jobs=-1,
    verbose=1,
)

om4, ppa4 = run_trial(
    stack_name_4, stack4,
    x_train_scaled, y_train,
    x_test_raw, x_test_scaled, y_test
)



In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# SUMMARY — print all trial results side by side
# Run after all trials complete
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("STACKING TRIALS SUMMARY")
print("="*70)
print(f"{'Trial':<40} {'Acc':>7} {'AUC':>7} {'F1':>7} {'MCC':>7} {'Thresh':>8}")
print("-"*70)

trial_results = [
    ("Stack_RF_XGB_LGBM__LR",              models.get("Stack_RF_XGB_LGBM__LR")),
    (stack_name_2,                          models.get(stack_name_2)),
    (stack_name_3,                          models.get(stack_name_3)),
    (stack_name_4,                          models.get(stack_name_4)),
]

for name, model in trial_results:
    if model is None:
        print(f"{name:<40} {'skipped':>7}")
        continue
    y_pred_t, y_prob_t = predict(name, model, x_test_raw, x_test_scaled)
    om_t  = overall_metrics(y_test, y_pred_t, y_prob_t)
    ppa_t = per_payload_acc(name, model)
    thresh = detect_threshold(ppa_t)
    thresh_str = f"{thresh:.2f}" if not np.isnan(thresh) else "N/A"
    print(f"{name:<40} {om_t['Accuracy']:>7.4f} {om_t['ROC_AUC']:>7.4f} "
          f"{om_t['F1_Macro']:>7.4f} {om_t['MCC']:>7.4f} {thresh_str:>8}")

print("="*70)

In [ ]:
df1=pd.read_csv("/kaggle/working/results/per_payload_accuracy.csv")

In [ ]:
# ── Final Cell — Per-Payload Accuracy Per Model (Individual) ──────────────────

all_models = {
    "LR":                           models.get("LR"),
    "RF":                           models.get("RF"),
    "XGB":                          models.get("XGB"),
    "LGBM":                         models.get("LGBM"),
    "MLP":                          models.get("MLP"),
    "Stack_RF_XGB_LGBM__LR":        models.get("Stack_RF_XGB_LGBM__LR"),
    "Stack_RF_XGB_LGBM_MLP__LR":    models.get("Stack_RF_XGB_LGBM_MLP__LR"),
    "Stack_RF_XGB_LGBM__XGB":       models.get("Stack_RF_XGB_LGBM__XGB"),
    "Stack_RF_XGB_LGBM__LR_passthrough": models.get("Stack_RF_XGB_LGBM__LR_passthrough"),
}

for model_name, model in all_models.items():
    if model is None:
        print(f"\n{model_name}: skipped\n")
        continue

    ppa = per_payload_acc(model_name, model)

    print(f"\n{'='*45}")
    print(f"  {model_name}")
    print(f"{'='*45}")
    print(f"  {'Payload (bpp)':<18} {'Accuracy':>10}")
    print(f"  {'-'*30}")

    for payload in sorted(ppa.keys()):
        acc = ppa[payload]
        bar = '█' * int(acc * 20)    # visual bar scaled to 20 chars = 100%
        print(f"  {payload:<18.2f} {acc:>10.4f}  {bar}")

    print(f"\n  Detection threshold : {detect_threshold(ppa):.2f} bpp")
    print(f"  Low  band mean (0.05–0.50) : {band_mean(ppa, 0.05, 0.50):.4f}")
    print(f"  Mid  band mean (0.55–1.00) : {band_mean(ppa, 0.55, 1.00):.4f}")
    print(f"  High band mean (1.05–2.00) : {band_mean(ppa, 1.05, 2.00):.4f}")

    # Save individual CSV
    df = pd.DataFrame({
        "Payload":  sorted(ppa.keys()),
        "Accuracy": [ppa[p] for p in sorted(ppa.keys())]
    })
    path = os.path.join(RESULTS_DIR, f"{model_name}_payload_accuracy.csv")
    df.to_csv(path, index=False)
    print(f"  Saved: {path}")

In [ ]:
# ── Read Per-Payload CSVs and Plot All Models ──────────────────────────────────

import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

best_model = None
best_score = -1

fig, ax = plt.subplots(figsize=(14, 7))

for i, name in enumerate(model_names):
    path = os.path.join(RESULTS_DIR, f"{name}_payload_accuracy.csv")
    if not os.path.exists(path):
        print(f"Skipped (file not found): {path}")
        continue

    df = pd.read_csv(path)

    # 🔹 Convert accuracy to %
    df["Accuracy"] = df["Accuracy"] * 100

    # 🔹 Track best model (use mean accuracy across payloads)
    mean_acc = df["Accuracy"].mean()
    if mean_acc > best_score:
        best_score = mean_acc
        best_model = name

    ls, marker = styles[i]
    ax.plot(
        df["Payload"], df["Accuracy"],
        linestyle=ls, marker=marker, markersize=4,
        color=colors[i], linewidth=1.8,
        label=name,
    )

# 🔹 Threshold now in %
ax.axhline(70, linestyle="--", color="black", linewidth=1.2, label="70% threshold")

ax.set_xlabel("Payload (bpp)", fontsize=13)
ax.set_ylabel("Accuracy (%)", fontsize=13)
ax.set_title("Per-Payload Detection Accuracy — All Models", fontsize=14, fontweight="bold")
ax.set_ylim(45, 100)
ax.set_xticks([round(x * 0.1, 1) for x in range(0, 21)])
ax.tick_params(axis="x", rotation=45)
ax.grid(alpha=0.25)
ax.legend(loc="upper left", fontsize=9, framealpha=0.9)

fig.tight_layout()
plot_path = os.path.join(RESULTS_DIR, "all_models_per_payload.png")
fig.savefig(plot_path, dpi=220)
plt.show()

print(f"Saved: {plot_path}")

# 🔹 Final result
print(f"\nBest Model: {best_model}")
print(f"Average Accuracy: {best_score:.2f}%")

In [ ]:
df_tops=pd.read_csv("/kaggle/working/results/per_payload_accuracy.csv")

In [ ]:
print(df_tops.iloc[0])

In [ ]:
import pandas as pd

# 🔹 Load
df_tops = pd.read_csv("/kaggle/working/results/per_payload_accuracy.csv")

# 🔹 Clean column names
df_tops.columns = df_tops.columns.str.strip()

# 🔹 Clean and convert all columns
for col in df_tops.columns:
    df_tops[col] = (
        df_tops[col]
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.strip()
    )
    df_tops[col] = pd.to_numeric(df_tops[col], errors="coerce")

# 🔹 Separate Payload
df_tops.rename(columns={df_tops.columns[0]: "Payload"}, inplace=True)
df_tops["Payload"] = pd.to_numeric(df_tops["Payload"], errors="coerce")

# 🔹 Convert to % if needed
numeric_cols = df_tops.columns.drop("Payload")
if df_tops[numeric_cols].max().max() <= 1:
    df_tops[numeric_cols] = df_tops[numeric_cols] * 100

# 🔹 Set index
df_tops = df_tops.set_index("Payload")

# 🔹 Work ONLY with numeric columns
numeric_df = df_tops.select_dtypes(include=["number"])

# 🔹 Best per payload
df_tops["Best_Model"] = numeric_df.idxmax(axis=1)
df_tops["Best_Accuracy"] = numeric_df.max(axis=1)

# 🔹 Final result table
result = df_tops[["Best_Model", "Best_Accuracy"]].reset_index()
result = result.sort_values(by="Payload")

print("\nPer-Payload Best Model:\n")
print(result.to_string(index=False))

# 🔹 Overall best model (mean performance)
overall_best = numeric_df.mean().idxmax()
overall_score = numeric_df.mean().max()

print(f"\nOverall Best Model: {overall_best}")
print(f"Average Accuracy: {overall_score:.2f}%")

In [ ]:
# MLP Neural Network
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu",
    solver="adam",
    batch_size=128,
    learning_rate="adaptive",
    max_iter=400,          # increased from 100
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=50,   # increased from 10 — more patient
    tol=1e-4,              # tighter than default 1e-4
    random_state=RANDOM_STATE,
    verbose=True,
)
mlp.fit(x_train_scaled, y_train)
joblib.dump(mlp, os.path.join(MODELS_DIR, "mlp_model.joblib"))
models["MLP"] = mlp
print("Trained: MLP Neural Network")

In [ ]:
"""
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu",
    solver="adam",

    batch_size=128,
    learning_rate="constant",
    learning_rate_init=0.001,

    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,

    n_iter_no_change=50,   # 🔥 key change
    tol=1e-4,              # 🔥 less strict

    alpha=1e-4,
    random_state=RANDOM_STATE,
    verbose=True,
)
"""
# --> Iteration 129, loss = 0.33919903
# Validation score: 0.805121
# Validation score did not improve more than tol=0.000010 for 30 consecutive epochs. Stopping.
# Trained: MLP Neural Network

In [ ]:
mlp = joblib.load(os.path.join(MODELS_DIR, "mlp_model.joblib"))

In [ ]:
# ── MLP Per-Payload Evaluation ────────────────────────────────────────────────

model_name = "MLP"
model = models.get("MLP")

if model is None:
    print("MLP model not found!")
else:
    ppa = per_payload_acc(model_name, model)

    print(f"\n{'='*45}")
    print(f"  {model_name}")
    print(f"{'='*45}")
    print(f"  {'Payload (bpp)':<18} {'Accuracy':>10}")
    print(f"  {'-'*30}")

    for payload in sorted(ppa.keys()):
        acc = ppa[payload]
        bar = '█' * int(acc * 20)
        print(f"  {payload:<18.2f} {acc:>10.4f}  {bar}")

    print(f"\n  Detection threshold : {detect_threshold(ppa):.2f} bpp")
    print(f"  Low  band mean (0.05–0.50) : {band_mean(ppa, 0.05, 0.50):.4f}")
    print(f"  Mid  band mean (0.55–1.00) : {band_mean(ppa, 0.55, 1.00):.4f}")
    print(f"  High band mean (1.05–2.00) : {band_mean(ppa, 1.05, 2.00):.4f}")

    # 🔹 Save CSV (same format as others)
    df = pd.DataFrame({
        "Payload":  sorted(ppa.keys()),
        "Accuracy": [ppa[p] for p in sorted(ppa.keys())]
    })

    path = os.path.join(RESULTS_DIR, f"{model_name}_payload_accuracy.csv")
    df.to_csv(path, index=False)

    print(f"\nSaved: {path}")

In [ ]:
# ── Detection Coverage vs LSB Embedding Methods (with PSNR/SSIM) ─────────────

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import os

mlp_df  = pd.read_csv(os.path.join(RESULTS_DIR, "MLP_payload_accuracy.csv"))
lgbm_df = pd.read_csv(os.path.join(RESULTS_DIR, "LGBM_payload_accuracy.csv"))

mlp_acc  = dict(zip(mlp_df["Payload"].round(2),  mlp_df["Accuracy"]))
lgbm_acc = dict(zip(lgbm_df["Payload"].round(2), lgbm_df["Accuracy"]))

def best_acc(payload):
    nearest = round(round(payload / 0.05) * 0.05, 2)
    nearest = min(2.0, max(0.05, nearest))
    return max(mlp_acc.get(nearest, 0), lgbm_acc.get(nearest, 0))

# Embedding methods — now with PSNR and SSIM from the table
# Format: (name, payload, label, psnr, ssim)
# None = not available in the source table
methods = [
    ("Saliency-based adaptive LSB\n(Sci. Reports 2025)",   0.25, "0.1–0.4", 58.20, 0.9979),
    ("GLM/MLE LSB\n(Muhammad et al. 2015)",                0.85, "0.8–0.9", 50.50, 0.9990),
    ("Classical 1-LSB\n(Generic baseline)",                1.00, "1.0",     51.00, 0.9990),
    ("CISSKA-LSB\n(Muhammad et al. 2017)",                 1.00, "~1.0",    49.50, 0.9990),
    ("LSB Matching Revisited\n(Mielikainen 2006)",         1.00, "~1.0",    51.00, 0.9990),
    ("Block determinant LSB\n(Shehzad & Dag 2019)",        1.00, "~1.0",    49.50, None),
    ("Edge-adaptive LSB\n(Gaurav & Ghanekar 2018)",        1.25, "~1.25",   47.00, 0.9970),
    ("Edge-area dilation LSB\n(Setiadi 2019)",             1.42, "~1.42",   47.00, 0.9977),
    ("Enhanced Canny-Sobel LSB\n(Setiadi & Jumanto 2018)", 2.10, "~2.1",    41.50, 0.9965),
    ("Hybrid edge LSB\n(Chen et al. 2010)",                2.10, "~2.1",    40.00, None),
    ("Multi-bit LSB k=2-5\n(Generic k-LSB)",               2.00, "2.0–5.0", 37.50, 0.9700),
    ("LSB + PVD hybrid\n(Swain 2016)",                     3.13, "~3.1",    38.50, None),
    ("Adaptive PVD + LSB\n(Khodaei et al. 2016)",          3.42, "~3.4",    39.00, None),
    ("Data mapping + LSB\n(Zakaria et al. 2018)",          3.23, "3.14–3.33",39.00, None),
    ("Four-pixel diff + LSB\n(Liao et al. 2011)",          3.13, "~3.13",   39.00, None),
    ("Hash-driven 4-LSB\n(Ghadi et al. 2023)",             4.00, "4.0",     None,  None),
]

names    = [m[0] for m in methods]
payloads = [m[1] for m in methods]
labels   = [m[2] for m in methods]
psnrs    = [m[3] for m in methods]
ssims    = [m[4] for m in methods]
accs     = [best_acc(min(p, 2.0)) * 100 for p in payloads]
colors   = ["#d62728" if a < 70 else "#2ca02c" for a in accs]

# ── 3-panel figure ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(
    1, 3, figsize=(20, 9),
    gridspec_kw={"width_ratios": [3, 1, 1]}
)

# Panel 1 — Detection accuracy bar chart
ax1 = axes[0]
bars = ax1.barh(names, accs, color=colors, edgecolor="white", height=0.65)
for bar, acc, pl in zip(bars, accs, labels):
    ax1.text(
        bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
        f"{acc:.1f}%  [{pl} bpp]",
        va="center", ha="left", fontsize=8.5, color="#333333"
    )
ax1.axvline(70, linestyle="--", color="black", linewidth=1.5)
ax1.set_xlabel("Detection Accuracy (%)", fontsize=11)
ax1.set_title("Detection Accuracy", fontsize=12, fontweight="bold")
ax1.set_xlim(40, 108)
ax1.grid(axis="x", alpha=0.25)
detected = mpatches.Patch(color="#2ca02c", label="Detected (≥70%)")
missed   = mpatches.Patch(color="#d62728", label="Below threshold (<70%)")
ax1.legend(handles=[detected, missed], loc="lower right", fontsize=9)

# Panel 2 — PSNR
ax2 = axes[1]
psnr_vals  = [p if p is not None else 0 for p in psnrs]
psnr_avail = [p is not None for p in psnrs]
psnr_colors = ["#3266ad" if av else "#cccccc" for av in psnr_avail]
ax2.barh(names, psnr_vals, color=psnr_colors, edgecolor="white", height=0.65)
for i, (val, avail) in enumerate(zip(psnr_vals, psnr_avail)):
    label = f"{val:.0f} dB" if avail else "N/A"
    ax2.text(val + 0.3, i, label, va="center", ha="left", fontsize=8, color="#333333")
ax2.axvline(40, linestyle=":", color="orange", linewidth=1.2, label="40 dB threshold")
ax2.set_xlabel("PSNR (dB)", fontsize=11)
ax2.set_title("Image Quality\n(PSNR)", fontsize=12, fontweight="bold")
ax2.set_xlim(0, 75)
ax2.set_yticks([])
ax2.grid(axis="x", alpha=0.25)
ax2.legend(fontsize=8, loc="lower right")

# Panel 3 — SSIM
ax3 = axes[2]
ssim_vals  = [s if s is not None else 0 for s in ssims]
ssim_avail = [s is not None for s in ssims]
ssim_colors = ["#9467bd" if av else "#cccccc" for av in ssim_avail]
ax3.barh(names, ssim_vals, color=ssim_colors, edgecolor="white", height=0.65)
for i, (val, avail) in enumerate(zip(ssim_vals, ssim_avail)):
    label = f"{val:.4f}" if avail else "N/A"
    ax3.text(val + 0.0002, i, label, va="center", ha="left", fontsize=8, color="#333333")
ax3.axvline(0.99, linestyle=":", color="orange", linewidth=1.2, label="0.99 threshold")
ax3.set_xlabel("SSIM", fontsize=11)
ax3.set_title("Image Quality\n(SSIM)", fontsize=12, fontweight="bold")
ax3.set_xlim(0.93, 1.003)
ax3.set_yticks([])
ax3.grid(axis="x", alpha=0.25)
ax3.legend(fontsize=8, loc="lower right")

fig.suptitle(
    "Detection Coverage vs Published LSB Embedding Methods\n"
    "with Image Quality Metrics (PSNR / SSIM)",
    fontsize=13, fontweight="bold", y=1.01
)
fig.tight_layout()
path = os.path.join(RESULTS_DIR, "detection_coverage_with_quality.png")
fig.savefig(path, dpi=220, bbox_inches="tight")
plt.show()
print(f"Saved: {path}")

# ── Summary table ─────────────────────────────────────────────────────────────
print("\nDetection Coverage Summary")
print("="*80)
print(f"{'Method':<42} {'bpp':>6} {'PSNR':>8} {'SSIM':>8} {'Det. Acc':>10} {'':>3}")
print("-"*80)
for name, payload, label, psnr, ssim, acc in zip(names, payloads, labels, psnrs, ssims, accs):
    psnr_str = f"{psnr:.0f} dB" if psnr else "N/A"
    ssim_str = f"{ssim:.4f}"    if ssim else "N/A"
    status   = "✓" if acc >= 70 else "✗"
    clean    = name.replace("\n", " ")
    print(f"{clean:<42} {label:>6} {psnr_str:>8} {ssim_str:>8} {acc:>8.1f}%  {status}")
print("="*80)
detected_count = sum(1 for a in accs if a >= 70)
print(f"\nDetected: {detected_count}/{len(methods)} methods above 70% threshold")
print("Grey bars = PSNR/SSIM not reported in source paper")

In [ ]:
# ── Cell 1: Your Per-Payload Curve vs SOTA Reference Lines ───────────────────

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import os

mlp_df  = pd.read_csv(os.path.join(RESULTS_DIR, "MLP_payload_accuracy.csv"))
lgbm_df = pd.read_csv(os.path.join(RESULTS_DIR, "LGBM_payload_accuracy.csv"))

mlp_acc  = dict(zip(mlp_df["Payload"].round(2),  mlp_df["Accuracy"] * 100))
lgbm_acc = dict(zip(lgbm_df["Payload"].round(2), lgbm_df["Accuracy"] * 100))
best_per_payload = {
    p: max(mlp_acc.get(p, 0), lgbm_acc.get(p, 0))
    for p in mlp_acc
}
payloads_sorted = sorted(best_per_payload.keys())
accs_sorted     = [best_per_payload[p] for p in payloads_sorted]

sota = [
    ("Chi-Square Attack\n(Westfeld & Pfitzmann 2000)",  75,  "classical"),
    ("RS Analysis\n(Fridrich et al. 2001)",              80,  "classical"),
    ("Sample Pair Analysis\n(Dumitrescu et al. 2003)",   82,  "classical"),
    ("SPAM Features\n(Pevny et al. 2010)",               90,  "ml"),
    ("Ensemble Classifiers\n(Kodovsky et al. 2012)",     92,  "ml"),
    ("Stacked Ensemble 8-feat\n(Miranda 2025)",          93.29,"ml"),
    ("CNN-based\n(Xu et al. 2016)",                      93,  "dl"),
    ("Attention-CNN\n(Tan et al. 2021)",                 93.29,"dl"),
    ("SRNet\n(Boroumand et al. 2019)",                   96,  "dl"),
]

type_colors = {
    "classical": "#E07B39",
    "ml":        "#3266ad",
    "dl":        "#9467bd",
}

# Spread label positions evenly to avoid overlap
label_positions = np.linspace(44, 98, len(sota))

fig, ax = plt.subplots(figsize=(16, 8))

ax.plot(payloads_sorted, accs_sorted, color="#2ca02c", linewidth=2.5,
        marker="o", markersize=3.5, label="Your Best Model (MLP/LGBM)", zorder=5)

ax.axvspan(0.05, 0.50, alpha=0.08, color="red", label="Prior work payload range")

for (name, acc, stype), label_y in zip(sota, label_positions):
    color = type_colors[stype]
    # Actual reference line at true accuracy
    ax.axhline(acc, linestyle=":", linewidth=1.0, color=color, alpha=0.5)
    # Annotation with leader line from label_y to true acc at x=2.05
    ax.annotate(
        name,
        xy=(2.05, acc),
        xytext=(2.20, label_y),
        fontsize=7.5,
        color=color,
        va="center",
        arrowprops=dict(arrowstyle="-", color=color, alpha=0.5, lw=0.8),
    )
    # Dot at the line endpoint
    ax.plot(2.05, acc, "o", color=color, markersize=4, alpha=0.8)

ax.axhline(70, linestyle="--", color="black", linewidth=1.3)

ax.set_xlabel("Payload (bpp)", fontsize=12)
ax.set_ylabel("Detection Accuracy (%)", fontsize=12)
ax.set_title("Your Per-Payload Detection Accuracy vs Published Detector Benchmarks",
             fontsize=13, fontweight="bold")
ax.set_xlim(0, 4.5)
ax.set_ylim(40, 100)
ax.grid(alpha=0.2)

legend_handles = [
    plt.Line2D([0],[0], color="#2ca02c", linewidth=2.5,
               label="Your model (MLP/LGBM)"),
    plt.Line2D([0],[0], color="black", linewidth=1.3,
               linestyle="--", label="70% threshold"),
    mpatches.Patch(color="#E07B39", alpha=0.8, label="Classical detectors"),
    mpatches.Patch(color="#3266ad", alpha=0.8, label="ML detectors"),
    mpatches.Patch(color="#9467bd", alpha=0.8, label="Deep learning detectors"),
    mpatches.Patch(color="red",     alpha=0.15, label="Prior work payload range"),
]
ax.legend(handles=legend_handles, fontsize=9, loc="upper left")

fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "sota_curve_comparison.png"),
            dpi=220, bbox_inches="tight")
plt.show()
print("Saved: sota_curve_comparison.png")

In [ ]:
# ── Cell 2: Aggregate Accuracy Bar Chart — Your Model vs Published Detectors ──

your_low_band = np.mean([
    best_per_payload.get(round(i * 0.05, 2), 0)
    for i in range(2, 11)   # 0.10 to 0.50
])

sota_entries = [
    ("Chi-Square Attack\n(Westfeld & Pfitzmann 2000)",  70,   80,   "classical"),
    ("RS Analysis\n(Fridrich et al. 2001)",              75,   85,   "classical"),
    ("Sample Pair Analysis\n(Dumitrescu et al. 2003)",   80,   85,   "classical"),
    ("SPAM Features\n(Pevny et al. 2010)",               90,   90,   "ml"),
    ("Ensemble Classifiers\n(Kodovsky et al. 2012)",     92,   92,   "ml"),
    ("Stacked Ensemble 8-feat\n(Miranda 2025)",          93.29,93.29,"ml"),
    ("CNN-based\n(Xu et al. 2016)",                      93,   93,   "dl"),
    ("Attention-CNN\n(Tan et al. 2021)",                 93.29,93.29,"dl"),
    ("SRNet\n(Boroumand et al. 2019)",                   96,   96,   "dl"),
    ("Your Model\n(avg 0.1–0.5 bpp)",                   your_low_band, your_low_band, "yours"),
]

type_colors = {
    "classical": "#E07B39",
    "ml":        "#3266ad",
    "dl":        "#9467bd",
    "yours":     "#2ca02c",
}

names  = [e[0] for e in sota_entries]
lows   = [e[1] for e in sota_entries]
highs  = [e[2] for e in sota_entries]
colors = [type_colors[e[3]] for e in sota_entries]

fig, ax = plt.subplots(figsize=(13, 8))

for i, (lo, hi, color, name) in enumerate(zip(lows, highs, colors, names)):
    ax.barh(i, hi, color=color, edgecolor="white", height=0.6, alpha=0.85)
    if lo != hi:
        ax.barh(i, lo, color="white", edgecolor="white", height=0.6)
        label_text = f"{lo:.0f}–{hi:.0f}%"
    else:
        label_text = f"{hi:.1f}%"
    ax.text(hi + 0.4, i, label_text, va="center", fontsize=10, color="#333333")

ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=10)
ax.axvline(70, linestyle="--", color="black", linewidth=1.3, label="70% threshold")
ax.set_xlabel("Reported Detection Accuracy (%)", fontsize=12)
ax.set_title(
    "Aggregate Accuracy: Your Model vs Published Detectors\n"
    "(Your model evaluated at 0.1–0.5 bpp to match prior work's test range)",
    fontsize=12, fontweight="bold"
)
ax.set_xlim(50, 108)
ax.grid(axis="x", alpha=0.2)

legend_handles = [
    mpatches.Patch(color="#E07B39", label="Classical detectors"),
    mpatches.Patch(color="#3266ad", label="ML detectors"),
    mpatches.Patch(color="#9467bd", label="Deep learning detectors"),
    mpatches.Patch(color="#2ca02c", label="Your model"),
    plt.Line2D([0],[0], color="black", linewidth=1.3,
               linestyle="--", label="70% threshold"),
]
ax.legend(handles=legend_handles, fontsize=9, loc="lower right")

fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "sota_bar_comparison.png"), dpi=220, bbox_inches="tight")
plt.show()

print(f"Saved: sota_bar_comparison.png")
print(f"\nYour model average accuracy:")
print(f"  0.10–0.50 bpp : {your_low_band:.1f}%")
print(f"  0.55–1.00 bpp : {np.mean([best_per_payload.get(round(i*0.05,2), 0) for i in range(11,21)]):.1f}%")
print(f"  1.05–2.00 bpp : {np.mean([best_per_payload.get(round(i*0.05,2), 0) for i in range(21,41)]):.1f}%")

In [ ]:
mlp_df  = pd.read_csv(os.path.join(RESULTS_DIR, "MLP_payload_accuracy.csv"))
lgbm_df = pd.read_csv(os.path.join(RESULTS_DIR, "LGBM_payload_accuracy.csv"))

for name, df in [("MLP", mlp_df), ("LGBM", lgbm_df)]:
    val = df.loc[df["Payload"].round(2) == 0.40, "Accuracy"].values[0] * 100
    print(f"{name} at 0.40 bpp: {val:.2f}%")

In [ ]:
# ── Low-payload specialist MLP ────────────────────────────────────────────────
from sklearn.neural_network import MLPClassifier

# Filter training data to low payloads only
low_payload_mask = (train_balanced["Payload"] <= 0.50) | (train_balanced["Payload"] == 0.0)
train_low = train_balanced[low_payload_mask]
print(f"Low-payload training set: {len(train_low):,} rows")

x_low, y_low = to_xy(train_low)
x_low_scaled = scaler.transform(x_low)

mlp_low = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64, 32),
    activation="relu",
    solver="adam",
    batch_size=128,
    learning_rate="constant",
    learning_rate_init=0.001,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=50,
    tol=1e-4,
    alpha=1e-4,
    random_state=RANDOM_STATE,
    verbose=True,
)
mlp_low.fit(x_low_scaled, y_low)
joblib.dump(mlp_low, os.path.join(MODELS_DIR, "mlp_low_payload.joblib"))
models["MLP_Low"] = mlp_low
print("Trained: MLP Low-Payload Specialist")

# Evaluate only on low payloads
low_test = test_df[(test_df["Payload"] <= 0.50) | (test_df["Payload"] == 0.0)]
clean_pool = low_test[low_test["Payload"] == 0.0]

print("\nLow-payload specialist results:")
print(f"  {'Payload':<10} {'Accuracy':>10}")
print(f"  {'-'*22}")
for p in sorted([float(x) for x in low_test["Payload"].unique() if x > 0]):
    stego_sub = low_test[low_test["Payload"] == p]
    clean_sub = clean_pool.sample(n=len(stego_sub), random_state=RANDOM_STATE)
    sub = pd.concat([clean_sub, stego_sub])
    xr, yt = to_xy(sub)
    xs = scaler.transform(xr)
    yp = mlp_low.predict(xs)
    acc = accuracy_score(yt, yp) * 100
    print(f"  {p:<10.2f} {acc:>9.2f}%")